# 04. SQL Joins: Inner, Left, Right, Full Outer, Cross, Self & Semi/Anti Joins

### 📝 Universal SQL Execution Order (All SQL Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline (All 12 Clauses) ────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. QUALIFY         ➔ 8. SELECT & CASE      ➔ 9. DISTINCT (Dedup)           │
│ ➔ 10. UNION/INTERSECT➔ 11. ORDER BY (Sort)   ➔ 12. LIMIT / OFFSET (Page)     │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **04. SQL Joins Masterclass**. Relational algebra enables combining independent relations horizontally through key constraints. This notebook provides comprehensive hands-on coverage of all join archetypes: matching intersecting tuples (`INNER JOIN`), preserving left/right baseline datasets (`LEFT OUTER JOIN`, `RIGHT OUTER JOIN`), bidirectional preservation (`FULL OUTER JOIN`), Cartesian permutations (`CROSS JOIN`), intra-table hierarchical pairing (`SELF JOIN`), and conditional existence filtering (`SEMI JOIN` via `EXISTS` & `ANTI JOIN` via `NOT EXISTS`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Intersecting Key Matching: `INNER JOIN ... ON`
- [x] 🔹 Left Baseline Preservation: `LEFT OUTER JOIN ... ON`
- [x] 🔹 Right Baseline Preservation: `RIGHT OUTER JOIN ... ON`
- [x] 🔹 Full Bidirectional Preservation: `FULL OUTER JOIN ... ON`
- [x] 🔹 Cartesian Product: `CROSS JOIN`
- [x] 🔹 Intra-Table Graph & Pair Traversal: `SELF JOIN`
- [x] 🔹 Semi-Join Existence Filtering: `EXISTS (...)`
- [x] 🔹 Anti-Join Exclusion Filtering: `NOT EXISTS (...)`
- [x] 🔍 Scenario: Multi-Table Customer Dispute Attribution & Merchant Exposure Analysis


In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    statements = [s.strip() for s in query.split(';') if s.strip()]
    if not statements:
        return ""
    if len(statements) > 1 and statements[-1].upper().startswith(('SELECT', 'WITH', 'EXPLAIN')):
        script = ";\n".join(statements[:-1]) + ";"
        cur = conn.cursor()
        cur.executescript(script)
        conn.commit()
        return pd.read_sql_query(statements[-1], conn)
    elif query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Intersecting Records: `INNER JOIN`
- **What it does:** Evaluates join predicates across two relations and returns only tuples that satisfy the matching condition in both tables. Unmatched rows in either table are filtered out.
- **Syntax:** `SELECT t1.cols, t2.cols FROM table_1 t1 INNER JOIN table_2 t2 ON t1.key = t2.key`
- **Dataset Application & Code Demonstration:** Joins `transactions` with `customers` on `customer_id` to correlate purchases with account tiers.


In [2]:
%%sql
SELECT 
    t.transaction_id,
    t.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    c.account_tier,
    t.transaction_amount
FROM transactions t
INNER JOIN customers c ON t.customer_id = c.customer_id
LIMIT 5;


,transaction_id,customer_id,customer_name,account_tier,transaction_amount
0,TX109326,C55082,Priya Rodriguez,Standard,607.78
1,TX106376,C76616,Paul Anderson,VIP,1819.11
2,TX103301,C65296,John Silva,Platinum,64.08
3,TX110701,C42098,Daniel Thompson,Standard,1025.73
4,TX103284,C97782,Margaret Robinson,Silver,772.74


### 🔹 Left Baseline Preservation: `LEFT OUTER JOIN`
- **What it does:** Preserves all tuples from the left table; matching right table attributes are populated, while unmatched records are padded with `NULL`.
- **Syntax:** `SELECT t1.cols, t2.cols FROM table_1 t1 LEFT JOIN table_2 t2 ON t1.key = t2.key`
- **Dataset Application & Code Demonstration:** Joins `customers` with `transactions` to ensure every customer is preserved, regardless of whether they have made transactions.


In [3]:
%%sql
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    t.transaction_id,
    t.transaction_amount
FROM customers c
LEFT JOIN transactions t ON c.customer_id = t.customer_id
LIMIT 5;


,customer_id,customer_name,transaction_id,transaction_amount
0,C93810,Richard Sharma,TX103677,286.69
1,C93810,Richard Sharma,TX103724,442.73
2,C93810,Richard Sharma,TX104540,93.24
3,C93810,Richard Sharma,TX105403,1368.86
4,C93810,Richard Sharma,TX105559,1133.10


### 🔹 Right Baseline Preservation: `RIGHT OUTER JOIN`
- **What it does:** Preserves all tuples from the right table; matching left table attributes are populated, while unmatched records are padded with `NULL`.
- **Syntax:** `SELECT t1.cols, t2.cols FROM table_1 t1 RIGHT JOIN table_2 t2 ON t1.key = t2.key`
- **Key Equivalence:** `table_1 RIGHT JOIN table_2` is semantically equivalent to `table_2 LEFT JOIN table_1`.
- **Dataset Application & Code Demonstration:** Joins `disputes` with `transactions` preserving all transactions on the right side.


In [4]:
%%sql
SELECT 
    d.dispute_id,
    d.dispute_status,
    t.transaction_id,
    t.transaction_amount,
    t.card_type
FROM disputes d
RIGHT JOIN transactions t ON d.transaction_id = t.transaction_id
LIMIT 5;


,dispute_id,dispute_status,transaction_id,transaction_amount,card_type
0,DSP200000,Chargeback Reversed,TX106378,1459.15,MasterCard
1,DSP200001,Under Review,TX113063,NaN,Visa
2,DSP200002,Chargeback Reversed,TX107681,1733.55,Amex
3,DSP200003,Won - Customer,TX109805,939.37,Discover
4,DSP200004,Won - Customer,TX112629,1090.06,Visa


### 🔹 Bidirectional Preservation: `FULL OUTER JOIN`
- **What it does:** Retains all records from both relations. Matching records are fused together, while unmatched records from either table are padded with `NULL`.
- **Syntax:** `SELECT t1.cols, t2.cols FROM table_1 t1 FULL OUTER JOIN table_2 t2 ON t1.key = t2.key`
- **Dataset Application & Code Demonstration:** Joins `customers` and `disputes` to observe both disputed customers and dispute-free customers in a single unified view.


In [5]:
%%sql
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    d.dispute_id,
    d.disputed_amount,
    d.dispute_status
FROM customers c
FULL OUTER JOIN disputes d ON c.customer_id = d.customer_id
WHERE d.dispute_id IS NOT NULL OR c.customer_id LIKE 'C1000%'
LIMIT 5;


,customer_id,customer_name,dispute_id,disputed_amount,dispute_status
0,C93810,Richard Sharma,DSP201607,391.62,Won - Merchant
1,C93810,Richard Sharma,DSP203475,391.62,Arbitration
2,C93810,Richard Sharma,DSP203545,1133.10,Under Review
3,C24592,Mark Scott,DSP200226,1550.01,Arbitration
4,C24592,Mark Scott,DSP200710,11.86,Arbitration


### 🔹 Cartesian Product: `CROSS JOIN`
- **What it does:** Computes the Cartesian product of two relations, pairing every row from the left table with every row from the right table ($M \times N$ rows).
- **Syntax:** `SELECT * FROM table_1 CROSS JOIN table_2`
- **Dataset Application & Code Demonstration:** Generates all possible combinations of operating regions and payment card types.


In [6]:
%%sql
SELECT 
    r.region,
    c.card_type
FROM (SELECT DISTINCT region FROM transactions WHERE region IS NOT NULL) r
CROSS JOIN (SELECT DISTINCT card_type FROM transactions WHERE card_type IS NOT NULL) c
LIMIT 8;


,region,card_type
0,North,Visa
1,North,Amex
2,North,Discover
3,North,MasterCard
4,West,Visa
5,West,Amex
6,West,Discover
7,West,MasterCard


### 🔹 Intra-Table Graph & Pair Traversal: `SELF JOIN`
- **What it does:** Joins a relation to itself by creating two distinct aliases (`t1` and `t2`). Used to compare rows within the same table, build hierarchical parent-child trees, or detect related paired events.
- **Syntax:** `SELECT t1.col, t2.col FROM table_1 t1 INNER JOIN table_1 t2 ON t1.group_key = t2.group_key AND t1.id < t2.id`
- **Dataset Application & Code Demonstration:** Detects pairs of fraudulent transactions associated with the same customer.


In [7]:
%%sql
SELECT 
    t1.customer_id,
    t1.transaction_id AS first_tx_id,
    t1.transaction_amount AS first_amount,
    t2.transaction_id AS second_tx_id,
    t2.transaction_amount AS second_amount,
    t1.region
FROM transactions t1
INNER JOIN transactions t2 
    ON t1.customer_id = t2.customer_id 
   AND t1.transaction_id < t2.transaction_id
WHERE t1.is_fraud = 1 AND t2.is_fraud = 1
LIMIT 5;


,customer_id,first_tx_id,first_amount,second_tx_id,second_amount,region
0,C31178,TX107785,1806.44,TX109671,231.90,North
1,C62581,TX101929,1533.75,TX108542,1629.03,East
2,C62581,TX101929,1533.75,TX110312,1944.65,East
3,C52992,TX107007,1880.87,TX110658,1120.29,South
4,C72402,TX102289,1835.39,TX104411,1810.90,West


### 🔹 Semi-Join Filtering: `EXISTS (...)`
- **What it does:** Filters candidate rows from the primary table based on whether at least one matching tuple exists in a target subquery. Unlike `INNER JOIN`, a `SEMI JOIN` never duplicates primary table rows even when multiple matches exist.
- **Syntax:** `SELECT t1.* FROM t1 WHERE EXISTS (SELECT 1 FROM t2 WHERE t2.key = t1.key)`
- **Dataset Application & Code Demonstration:** Identifies customers who have at least one active dispute filed.


In [8]:
%%sql
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    c.account_tier,
    c.credit_score
FROM customers c
WHERE EXISTS (
    SELECT 1 
    FROM disputes d 
    WHERE d.customer_id = c.customer_id
)
LIMIT 5;


,customer_id,customer_name,account_tier,credit_score
0,C93810,Richard Sharma,Platinum,476.0
1,C24592,Mark Scott,VIP,396.0
2,C13278,Karen Walker,Standard,475.0
3,C46048,Barbara Wilson,Silver,810.0
4,C42098,Daniel Thompson,Standard,702.0


### 🔹 Anti-Join Optimization: Isolating Unmatched Rows (`NOT EXISTS`)
- **What it does:** Identifies rows in a primary relation that have **zero** corresponding matching records in a target relation.
- **Syntax:** `SELECT t1.* FROM t1 WHERE NOT EXISTS (SELECT 1 FROM t2 WHERE t2.key = t1.key)`
- **Dataset Application & Code Demonstration:** Finds customers who have zero recorded transactions in the system.


In [9]:
%%sql
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    c.account_tier
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 
    FROM transactions t 
    WHERE t.customer_id = c.customer_id
)
LIMIT 5;


,customer_id,customer_name,account_tier
0,C42738,Sandra Clark,VIP
1,C64934,Amara Gonzalez,VIP
2,C11068,Aarav Johnson,Silver
3,C62595,Emma Johnson,Platinum
4,C25907,Mark Kim,Standard


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.

### 🔍 Scenario: Q1: Multi-Table Financial Settlement Attribution
- **Objective:** Combine transactions, customers, and merchants in a single 3-table join to calculate merchant dispute liability and gross transaction volumes.
- **Approach:** Perform chained `INNER JOIN` operations and aggregate financial metrics per merchant.


In [10]:
%%sql
SELECT 
    m.merchant_id,
    m.merchant_name,
    m.category,
    COUNT(t.transaction_id) AS total_tx_count,
    ROUND(SUM(t.transaction_amount), 2) AS total_volume_usd
FROM merchants m
INNER JOIN transactions t ON m.merchant_id = t.merchant_id
GROUP BY m.merchant_id, m.merchant_name, m.category
ORDER BY total_volume_usd DESC
LIMIT 5;


,merchant_id,merchant_name,category,total_tx_count,total_volume_usd
0,M5128,Pulse Express,Food & Dining,40,50504.15
1,M6761,SilverLine Tech,Healthcare & Wellness,46,48765.54
2,M9618,Beacon Mart,Crypto & Digital Assets,45,46568.24
3,M5757,Global Hub,Grocery & Supermarket,40,45356.37
4,M6952,Nexus Retail,E-Commerce & Marketplaces,40,44978.49
